# 01 - FMA-small genre recall

Visualizes the v0 quality gate: recall@k where k-nearest-neighbors share the seed track's genre.

**Pre-req** (heavy):
1. `python scripts/download_fma.py` (~7.5 GB)
2. `make embed ROOT=data/raw/fma_small OUT=models/store/fma.parquet` (~hours on CPU)

If the store is already built, this notebook only does the eval pass (~1 minute).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

STORE = ROOT / "models" / "store" / "fma.parquet"
AUDIO = ROOT / "data" / "raw" / "fma_small"
META = ROOT / "data" / "raw" / "fma_metadata"
for p in [STORE, AUDIO, META]:
    print(p, "exists:", p.exists())

In [ ]:
import numpy as np
from eval.fma import align_store_to_fma, load_fma_index
from eval.metrics import recall_at_k_genre
from predictor.store import EmbeddingStore

store = EmbeddingStore.open(STORE)
fma = load_fma_index(AUDIO, META, subset="small")
joined = align_store_to_fma(store.df, fma)
print(f"store size: {len(store)}; joined to FMA: {len(joined)}")
joined["genre_top"].value_counts()

In [ ]:
matrix = np.stack([np.asarray(e, dtype=np.float32) for e in joined["embedding"].to_list()])
labels = joined["genre_top"].astype(str).to_list()
k_values = (1, 5, 10, 20)
recall = recall_at_k_genre(matrix, labels, k_values=k_values)
for k in k_values:
    print(f"recall@{k:>2} = {recall[k]:.3f}")
print("\nrandom baseline (8 balanced classes) = 0.125")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from eval.metrics import _dot_topk

genres = sorted(set(labels))
g_to_i = {g: i for i, g in enumerate(genres)}
conf = np.zeros((len(genres), len(genres)), dtype=np.float32)
K = 10
for i in range(matrix.shape[0]):
    nbrs = _dot_topk(matrix, matrix[i], k=K, exclude_self=i)
    for j in nbrs:
        conf[g_to_i[labels[i]], g_to_i[labels[j]]] += 1
conf /= conf.sum(axis=1, keepdims=True).clip(min=1)

plt.figure(figsize=(7, 6))
sns.heatmap(conf, xticklabels=genres, yticklabels=genres, annot=True, fmt=".2f", cmap="viridis")
plt.title(f"FMA-small @ K={K}: row=seed genre, col=neighbor genre")
plt.tight_layout()
plt.show()